[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/next-gen-aiam/blob/main/notebooks/04c_dl_policy_strategies_cuda.ipynb)

## Session 3d — Direct-Weight DL Portfolio Construction

Session 3d extends the deep learning portfolio construction work to the direct-weight paradigm. Building on the predict-then-allocate methodology of Sessions 3a–3c-full, this notebook implements parametric portfolio policies (Brandt, Santa-Clara & Valkanov 2009) with deep learning backbones (MLP, LSTM, Transformer) trained end-to-end on portfolio-level objectives (Sharpe, CRRA utility with γ=5, CRRA with shrinkage to equal-weight). The architecture extends the 2-asset attention-based demonstration in Cheng & Wu (J.P. Morgan, March 2024) to the same 29-asset universe and 17-feature panel used throughout Sessions 1–3.

**Environment-aware:** runs full 10-seed × 80-epoch experiment on CUDA (Colab Blackwell) or a fast smoke-test on CPU/MPS (local M4, 1 seed × 5 epochs × 9 configs). Companion to `04b_dl_strategies_cuda.ipynb` — do not modify that file.

## 1. Motivation and framework

### Predict-then-allocate vs direct-weight

Sessions 3a–3c follow a *predict-then-allocate* pipeline: a model first predicts one-step-ahead returns $\hat{r}_{i,t+1} = f_\theta(x_{i,t})$, then a separate allocation rule converts those forecasts into portfolio weights $w_t = g(\hat{r}_t)$. The two stages are trained independently, so the allocation step does not feed back into the forecast objective.

A *direct-weight* policy collapses both stages into a single mapping trained end-to-end:
$$w_{i,t} = f_\theta(x_{i,t})$$
The model is optimised directly on portfolio-level utility, so forecasting accuracy is replaced by allocation quality as the training signal.

### BSV optimisation objective

Brandt, Santa-Clara & Valkanov (2009) parameterise portfolio weights as a function of asset characteristics and maximise expected utility in-sample:

$$\max_\theta \frac{1}{T}\sum_{t=1}^{T} U\!\left(R_{p,t+1}\right), \qquad R_{p,t+1} = \sum_{i=1}^{N} w_{i,t}(\theta)\,r_{i,t+1}$$

where $U(\cdot)$ is a utility function and $w_{i,t}(\theta)$ are weights produced by the policy network. We replace their linear policy with MLP, LSTM, and Transformer backbones and treat the optimisation as a mini-batch stochastic gradient descent problem.¹

> ¹ Brandt, M. W., Santa-Clara, P., & Valkanov, R. (2009). Parametric portfolio policies: Exploiting characteristics in the cross-section of equity returns. *Review of Financial Studies*, 22(9), 3411–3447.

### Three loss variants

We train three variants of each architecture, differing only in the loss function:

**Sharpe loss** (return-to-vol ratio, predecessor of the full framework):
$$\mathcal{L}_\text{Sharpe}(\theta) = -\frac{\hat{\mu}(R_p)}{\hat{\sigma}(R_p)}$$

**CRRA utility loss** ($\gamma = 5$, the framework's central contribution):
$$\mathcal{L}_\text{CRRA}(\theta) = -\frac{1}{T}\sum_{t=1}^{T}\frac{(1 + R_{p,t})^{1-\gamma}}{1-\gamma}$$
The Taylor expansion of CRRA produces alternating signs on odd/even moments: the loss rewards higher returns and positive skewness while penalising variance and kurtosis, without explicitly estimating them.

**CRRA + shrinkage toward equal-weight** (framework extension):
$$w_{i,t} = f_\theta(x_{i,t})\cdot w_i^{\,b}, \quad w_i^{\,b} = 1/29 \text{ (EW benchmark)}$$
A sigmoid output bounds the multiplier to $[0,1]$, enforcing long-only shrinkage toward the equal-weight benchmark.²

> ² Brandt, M. W. (1999). Estimating portfolio and consumption choice: A conditional Euler equations approach. *Journal of Finance*, 54(5), 1609–1645.

### What this notebook adds to the Cheng & Wu (2024) demonstration

Cheng & Wu (2024) demonstrated direct-weight deep learning portfolio construction on a 2-asset (S&P 500 + 10Y Treasury) universe using attention-based models with 10-seed ensemble averaging.³ This notebook extends the approach to:

(a) **29 assets** (full cross-asset universe: equities, sector ETFs, international equity, fixed income, and commodities) matching Sessions 1–3 of this project, vs 2;
(b) **17 features** (10 numeric technical indicators + 7 asset-class one-hots) vs 5;
(c) **three architecture families** (MLP, LSTM, Transformer) compared in parallel under all 3 loss functions = 9 configs total;
(d) **direct comparison** against the 38-strategy baseline from Sessions 1–3, producing a 47-strategy unified comparison table.

> ³ Cheng, P., & Wu, E. (2024). Big Data and AI Strategies: Deep Learning Portfolio Construction. J.P. Morgan Global Quantitative & Derivatives Strategy, 14 March 2024.

In [ ]:
import sys, os, time, warnings
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print('Detected Colab — cloning repo...')
    os.system('git clone https://github.com/FranQuant/next-gen-aiam.git 2>&1 | tail -3')
    os.chdir('next-gen-aiam')
    print(f'Working dir: {os.getcwd()}')
    print('Installing dependencies (1-2 min)...')
    os.system('pip install -q -e . 2>&1 | tail -3')
    os.system('pip install -q xgboost pyarrow scipy scikit-learn cvxpy pandas-market-calendars matplotlib seaborn 2>&1 | tail -3')
else:
    print(f'Detected local environment — {os.getcwd()}')

import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    GPU_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    GPU_NAME = 'Apple Silicon MPS'
else:
    DEVICE = 'cpu'
    GPU_NAME = 'CPU'

print(f'Device: {DEVICE}  ({GPU_NAME})')
print(f'torch : {torch.__version__}')

os.environ.setdefault('OMP_NUM_THREADS', '1')

# Full: 10 seeds × 80 epochs (CUDA); smoke: 1 seed × 5 epochs (local M4)
if DEVICE == 'cuda':
    SEEDS_DL = tuple(range(10))
    ALL_CFG  = dict(max_epochs=80, patience=12)  # same config across all 9 for fair comparison
    RUN_MODE = 'full'
else:
    SEEDS_DL = (0,)
    ALL_CFG  = dict(max_epochs=5, patience=3)
    RUN_MODE = 'smoke'

print(f'\nRun mode : {RUN_MODE}')
print(f'Seeds    : {len(SEEDS_DL)}')
print(f'Epochs   : max_epochs={ALL_CFG["max_epochs"]}  patience={ALL_CFG["patience"]}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 10})
warnings.filterwarnings('ignore')

ROOT = Path(os.getcwd())
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

for p in [str(ROOT / 'src'), str(ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

from _shared import ann_sharpe, ann_return, ann_vol, max_drawdown, TRADING_DAYS
from aiam.data.panel import Panel
from aiam.data.split import TRAIN_END, TEST_START
from aiam.features.technical import (
    momentum, volatility, rsi, atr, bollinger, gap, volume_signal, forward_returns
)
from aiam.features.asset_class import asset_class_one_hot
from aiam.ml.workflow import chronological_splits, apply_standardizer
from aiam.dl.losses import sharpe_loss, crra_loss, crra_shrinkage_loss
from aiam.dl.policy_workflow import DirectWeightSeedEnsemble, _predict_policy
from aiam.strategy.dl_policy_strategies import (
    DirectWeightMLPStrategy,
    DirectWeightLSTMStrategy,
    DirectWeightTransformerStrategy,
)

print(f'ROOT     : {ROOT}')
print(f'Train end: {TRAIN_END}  |  Test start: {TEST_START}')

In [ ]:
returns = pd.read_parquet(ROOT / 'data/cache/returns_29_2003_2026.parquet')
returns.index = pd.to_datetime(returns.index)
returns.index.name = 'Date'
returns.columns.name = 'Asset'

prices = pd.read_parquet(ROOT / 'data/cache/prices_29.parquet')
prices.index = pd.to_datetime(prices.index)
prices.index.name = 'Date'
prices.columns.name = 'Asset'

ohlcv_raw = pd.read_parquet(ROOT / 'data/cache/prices_29_ohlcv_2003_2026.parquet')
ohlcv_raw.index = pd.MultiIndex.from_tuples(
    [(pd.Timestamp(d), t) for d, t in ohlcv_raw.index],
    names=['Date', 'Asset']
)

print(f'Returns: {returns.shape}, {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Prices:  {prices.shape}')

In [ ]:
rets = returns.copy()
prices_close  = ohlcv_raw['close'].unstack('Asset')
prices_open   = ohlcv_raw['open'].unstack('Asset')
prices_high   = ohlcv_raw['high'].unstack('Asset')
prices_low    = ohlcv_raw['low'].unstack('Asset')
prices_volume = ohlcv_raw['volume'].unstack('Asset')
ohlc_dict = {'open': prices_open, 'high': prices_high, 'low': prices_low,
             'close': prices_close, 'volume': prices_volume}

feat_mom_21   = momentum(rets, 21)
feat_mom_63   = momentum(rets, 63)
feat_mom_252  = momentum(rets, 252)
feat_vol_60   = volatility(rets, 60)
feat_vol_252  = volatility(rets, 252)
feat_rsi_14   = rsi(prices_close, 14)
feat_atr_raw  = atr(ohlc_dict, 14)
feat_atr_ratio = feat_atr_raw / prices_close
bb = bollinger(prices_close, window=20)
feat_bb_pct   = bb['pct']
feat_gap      = gap(ohlc_dict)
feat_vol_sig  = volume_signal(prices_volume, lookback=21)

numeric_frames = {
    'mom_21':        feat_mom_21,
    'mom_63':        feat_mom_63,
    'mom_252':       feat_mom_252,
    'vol_60':        feat_vol_60,
    'vol_252':       feat_vol_252,
    'rsi_14':        feat_rsi_14,
    'atr_14_ratio':  feat_atr_ratio,
    'bb_pct':        feat_bb_pct,
    'gap':           feat_gap,
    'vol_signal_21': feat_vol_sig,
}
panel_numeric = pd.concat({k: v.stack() for k, v in numeric_frames.items()}, axis=1)
panel_numeric.index.names = ['Date', 'Asset']

assets = rets.columns.tolist()
oh = asset_class_one_hot(assets)
feature_panel = panel_numeric.join(oh, on='Asset')
feature_panel = feature_panel.dropna(subset=['mom_252', 'vol_252'])

NUMERIC_COLS = list(numeric_frames.keys())
AC_COLS = list(oh.columns)
FEATURE_COLS = NUMERIC_COLS + AC_COLS

print(f'Feature panel: {feature_panel.shape}')
print(f'Feature cols ({len(FEATURE_COLS)}): {FEATURE_COLS}')

In [ ]:
HORIZON = 21
fwd_ret_wide = forward_returns(rets, HORIZON)
target_panel = fwd_ret_wide.stack()
target_panel.index.names = ['Date', 'Asset']
target_panel.name = 'target_21d'

common_idx = feature_panel.index.intersection(target_panel.dropna().index)
X_full = feature_panel.loc[common_idx]
y_full = target_panel.loc[common_idx]

panel_dates = X_full.index.get_level_values('Date').unique().sort_values()
train_dates, val_dates, test_dates = chronological_splits(
    panel_dates, train_end=TRAIN_END, test_start=TEST_START, validation_share=0.15
)

FAMILY_COLORS = {
    'Classical':                     '#1f4e79',
    'ML (single-fit)':               '#d62728',
    'ML (ensemble)':                 '#2ca02c',
    'DL (single-fit)':               '#9467bd',
    'DL (ensemble)':                 '#e377c2',
    'DL (weighted ensemble)':        '#f7b7d4',
    'DL (direct-weight)':            '#8c564b',
    'Walk-forward (default HPs)':    '#7f7f7f',
    'Walk-forward (val-optimal HPs)':'#bcbd22',
    'ML + VMP':                      '#17becf',
}

print(f'Target panel: {target_panel.shape}  X_full: {X_full.shape}')
print(f'Train:      {train_dates[0].date()} → {train_dates[-1].date()}  ({len(train_dates)} days)')
print(f'Validation: {val_dates[0].date()} → {val_dates[-1].date()}  ({len(val_dates)} days)')
print(f'Test:       {test_dates[0].date()} → {test_dates[-1].date()}  ({len(test_dates)} days)')

In [ ]:
# Equal-weight benchmark used as shrinkage prior for the CRRA+Shrink loss variant
benchmark_w_ew = np.ones(len(assets)) / len(assets)
print(f'Equal-weight benchmark: {len(assets)} assets, w_i = {benchmark_w_ew[0]:.4f}')
print('(Sigmoid output acts as per-asset multiplier on this benchmark in CRRA+Shrink.)')

In [ ]:
# Architecture → (strategy class, arch-specific hyperparams)
ARCH_CLASS = {
    'MLP':         (DirectWeightMLPStrategy,        {'hidden_dims': (32, 16), 'dropout': 0.10}),
    'LSTM':        (DirectWeightLSTMStrategy,        {'hidden_dim': 24,        'dropout': 0.10}),
    'Transformer': (DirectWeightTransformerStrategy, {'d_model': 32, 'nhead': 4, 'num_layers': 2, 'dropout': 0.10}),
}
# loss display name → loss_kind string accepted by _DLPolicyBase
LOSS_KIND = {'Sharpe': 'sharpe', 'CRRA': 'crra', 'CRRA+Shrink': 'crra_shrinkage'}

configs = [
    ('MLP', 'Sharpe'),   ('MLP', 'CRRA'),   ('MLP', 'CRRA+Shrink'),
    ('LSTM', 'Sharpe'),  ('LSTM', 'CRRA'),  ('LSTM', 'CRRA+Shrink'),
    ('Transformer', 'Sharpe'), ('Transformer', 'CRRA'), ('Transformer', 'CRRA+Shrink'),
]

strategies = {}
t_total_start = time.time()

for arch, loss in configs:
    name = f'DW({arch})[{loss}]'
    cls, arch_hp = ARCH_CLASS[arch]
    print(f'Training {name} ...', flush=True)
    t0 = time.time()
    strat = cls(
        feature_panel=X_full,
        target_panel=y_full,
        feature_cols=FEATURE_COLS,
        assets=assets,
        train_end=TRAIN_END,
        validation_share=0.15,
        seeds=SEEDS_DL,
        loss_kind=LOSS_KIND[loss],
        benchmark_w=benchmark_w_ew,  # used only for crra_shrinkage; ignored otherwise
        lr=1e-3,
        weight_decay=1e-4,
        batch_size=256,
        device=DEVICE,
        **arch_hp,
        **ALL_CFG,
    )
    strategies[name] = strat
    elapsed = time.time() - t0
    s = strat._seed_ensemble.fits[0].summary
    print(f'  {int(elapsed//60)}:{int(elapsed%60):02d}  seed0 best_epoch={s["best_epoch"]}  val_loss={s["best_val_loss"]:.4f}')

t_total = time.time() - t_total_start
print(f'\nAll 9 configs done in {int(t_total//60)}:{int(t_total%60):02d}')

In [ ]:
panel = Panel({'prices': prices, 'returns': returns})
eval_dates = returns.index[returns.index >= TEST_START]
test_idx   = returns.index[returns.index >= TEST_START]

def walk_forward_returns(strat, dates, returns_wide, panel_obj):
    ret_series = {}
    for d in dates[:-1]:
        w = strat.predict_weights(panel_obj, d)
        nxt = returns_wide.index[returns_wide.index > d]
        if len(nxt) == 0:
            continue
        r_next = returns_wide.loc[nxt[0]]
        ret_series[nxt[0]] = (w.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

print('Running walk-forward evaluation (9 configs)...')
dw_returns = {}
t0_wf = time.time()
for name, strat in strategies.items():
    r = walk_forward_returns(strat, eval_dates, returns, panel)
    dw_returns[name] = r
    sh = ann_sharpe(r.reindex(test_idx).dropna())
    print(f'  {name}: Sharpe={sh:.3f}')

dw_returns_df = pd.DataFrame(dw_returns).reindex(test_idx)
print(f'Walk-forward done ({time.time()-t0_wf:.0f}s total)')
print(f'Returns shape: {dw_returns_df.shape}')

In [ ]:
def per_seed_walk_forward(strat, seed_idx, eval_dates, returns_wide, panel_obj):
    """Walk-forward using a single seed's model (swaps ensemble temporarily)."""
    fr = strat._seed_ensemble.fits[seed_idx]
    orig_ensemble = strat._seed_ensemble
    orig_cache    = strat._weight_cache
    # Replace ensemble with single-seed version; rebuild weight cache
    strat._seed_ensemble = DirectWeightSeedEnsemble(fits=[fr], seeds=(fr.summary['seed'],))
    strat._weight_cache  = {}
    strat._cache_test_weights()
    ret = walk_forward_returns(strat, eval_dates, returns_wide, panel_obj)
    # Restore
    strat._seed_ensemble = orig_ensemble
    strat._weight_cache  = orig_cache
    return ret

n_seeds = len(SEEDS_DL)
print(f'Per-seed Sharpes ({n_seeds} seeds × 9 configs = {n_seeds*9} backtests)...')
seed_sharpes_by_config = {}
stability_rows = []
for name in list(strategies.keys()):
    strat = strategies[name]
    seed_sharpes = []
    for k in range(n_seeds):
        r = per_seed_walk_forward(strat, k, eval_dates, returns, panel)
        seed_sharpes.append(ann_sharpe(r.reindex(test_idx).dropna()))
    seed_sharpes_by_config[name] = list(seed_sharpes)  # explicit copy
    print(f'  {name}: mean={np.mean(seed_sharpes):.3f} ± {np.std(seed_sharpes):.3f}')
    stability_rows.append({
        'Strategy':         name,
        'Seeds':            len(seed_sharpes),
        'Min OOS Sharpe':   np.min(seed_sharpes),
        'Mean OOS Sharpe':  np.mean(seed_sharpes),
        'Max OOS Sharpe':   np.max(seed_sharpes),
        'Stdev OOS Sharpe': np.std(seed_sharpes),
    })

stability_table = pd.DataFrame(stability_rows).set_index('Strategy')
print('\nStability table:')
print(stability_table.round(4).to_string())

In [ ]:
baseline_path = ROOT / 'results/cuda/dl_strategies_comparison_3cfull.csv'
if baseline_path.exists():
    baseline = pd.read_csv(baseline_path).set_index('Strategy')
    print(f'3c-full baseline loaded: {len(baseline)} strategies')
else:
    ml_csv = ROOT / 'data/cache/portfolio_returns/ml_strategies_extended_v2_comparison.csv'
    baseline = pd.read_csv(ml_csv).set_index('Strategy')
    print(f'Fallback to ML baseline: {len(baseline)} strategies')

new_rows = []
for sname, r in dw_returns.items():
    r_test = r.reindex(test_idx).dropna()
    new_rows.append({
        'Strategy': sname,
        'Ann Ret':  ann_return(r_test),
        'Ann Vol':  ann_vol(r_test),
        'Sharpe':   ann_sharpe(r_test),
        'Max DD':   max_drawdown(r_test),
        'Family':   'DL (direct-weight)',
    })

new_df = pd.DataFrame(new_rows).set_index('Strategy')
comparison_3d = pd.concat([baseline, new_df]).sort_values('Sharpe', ascending=False)

out_dir = ROOT / 'results/cuda'
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'figures').mkdir(exist_ok=True)

comparison_3d.to_csv(out_dir / 'policy_strategies_comparison_3d.csv')
stability_table.to_csv(out_dir / 'policy_stability_table_3d.csv')
print(f'Combined: {len(comparison_3d)} strategies')
print(f'Saved: policy_strategies_comparison_3d.csv, policy_stability_table_3d.csv')
print('\nTop 12 strategies by Sharpe:')
print(comparison_3d[['Family', 'Ann Ret', 'Ann Vol', 'Sharpe', 'Max DD']].head(12).round(3).to_string())

In [ ]:
dw_returns_df.index.name = 'Date'
dw_returns_df.to_parquet(out_dir / 'policy_returns_3d.parquet')
print(f'Saved: policy_returns_3d.parquet  shape={dw_returns_df.shape}')

In [ ]:
fig_dir = out_dir / 'figures'

# Load reference returns for context lines
base_pub = pd.read_parquet(ROOT / 'data/published/strategy_returns_base.parquet')
base_pub.index = pd.to_datetime(base_pub.index)
vmp_pub = pd.read_parquet(ROOT / 'data/published/strategy_returns_vmp.parquet')
vmp_pub.index = pd.to_datetime(vmp_pub.index)
msr_lw_test  = base_pub['MSR(ledoit_wolf)'][base_pub.index >= TEST_START]
vmp_mdp_test = vmp_pub['VMP(MDP(ledoit_wolf))'][vmp_pub.index >= TEST_START]

dl_3cfull_path = ROOT / 'results/cuda/dl_returns_3cfull.parquet'
dl_returns_3cfull = pd.read_parquet(dl_3cfull_path) if dl_3cfull_path.exists() else pd.DataFrame()
ml_returns_ext = pd.read_parquet(ROOT / 'data/cache/portfolio_returns/ml_strategies_29assets_extended.parquet')

def get_returns(name):
    if name in dw_returns:
        return dw_returns[name]
    if not dl_returns_3cfull.empty and name in dl_returns_3cfull.columns:
        return dl_returns_3cfull[name]
    if name in ml_returns_ext.columns:
        return ml_returns_ext[name]
    if name == 'MSR(LW)':
        return msr_lw_test
    if name == 'VMP(MDP(LW))':
        return vmp_mdp_test
    return pd.Series(dtype=float)

ML_BAR = 2.579
DL_BEST_MEDIAN = 2.320  # MSR(MLP_μ̂) from 3c-full

# Plot 1: Equity curves — top-5 by Sharpe, ensure at least one DW
top5 = comparison_3d.head(5).index.tolist()
dw_rows = comparison_3d[comparison_3d['Family'] == 'DL (direct-weight)']
best_dw_name = dw_rows.index[0] if len(dw_rows) > 0 else None
plot_strats = top5 if (best_dw_name is None or best_dw_name in top5) else top5[:4] + [best_dw_name]

fig, ax = plt.subplots(figsize=(11, 5))
for sname in plot_strats:
    r = get_returns(sname).reindex(test_idx).dropna()
    if len(r) == 0:
        continue
    fam = comparison_3d.loc[sname, 'Family'] if sname in comparison_3d.index else 'DL (direct-weight)'
    color = FAMILY_COLORS.get(fam, '#333333')
    ax.plot((1 + r).cumprod().values, label=sname, color=color, lw=1.8)
ax.set_title('3d: Cumulative Wealth — Top strategies (test 2023–2026)')
ax.set_ylabel('Growth of $1')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / 'equity_curves_3d.png', dpi=150)
plt.show()

# Plot 2: Drawdown — same strategy set
fig, ax = plt.subplots(figsize=(11, 4))
for sname in plot_strats:
    r = get_returns(sname).reindex(test_idx).dropna()
    if len(r) == 0:
        continue
    wealth = (1 + r).cumprod()
    dd = wealth / wealth.cummax() - 1
    fam = comparison_3d.loc[sname, 'Family'] if sname in comparison_3d.index else 'DL (direct-weight)'
    color = FAMILY_COLORS.get(fam, '#333333')
    ax.fill_between(range(len(dd)), dd.values, 0, alpha=0.15, color=color)
    ax.plot(range(len(dd)), dd.values, label=sname, color=color, lw=1.2)
ax.set_title('3d: Underwater Drawdown (test period)')
ax.set_ylabel('Drawdown')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / 'drawdown_3d.png', dpi=150)
plt.show()

# Plot 3: Per-seed Sharpe distribution (9 boxplots)
cfg_names = list(strategies.keys())
short_labels = [
    n.replace('DW(MLP)[', 'MLP\n').replace('DW(LSTM)[', 'LSTM\n')
     .replace('DW(Transformer)[', 'Tfm\n').rstrip(']')
    for n in cfg_names
]
fig, ax = plt.subplots(figsize=(11, 4))
for i, name in enumerate(cfg_names):
    sharpes = seed_sharpes_by_config[name]
    ax.boxplot(sharpes, positions=[i], widths=0.4, patch_artist=True,
               boxprops=dict(facecolor='#c4a882', alpha=0.6),
               medianprops=dict(color='#8c564b', lw=2))
    ax.scatter([i] * len(sharpes), sharpes, alpha=0.7, color='#8c564b', s=40, zorder=5)
ax.set_xticks(range(len(cfg_names)))
ax.set_xticklabels(short_labels, fontsize=8)
ax.set_ylabel('OOS Sharpe')
ax.set_title(f'3d: Per-Seed OOS Sharpe ({n_seeds} seeds each)')
ax.axhline(ML_BAR, color='green', ls='--', lw=1.2, label=f'ML bar ({ML_BAR})')
ax.axhline(DL_BEST_MEDIAN, color='#9467bd', ls=':', lw=1.2, label=f'Best predict-then-wrap DL ({DL_BEST_MEDIAN})')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / 'seed_sharpe_dist_3d.png', dpi=150)
plt.show()

# Plot 4: Top-10 Sharpe bar chart
top10 = comparison_3d.head(10)
colors = [FAMILY_COLORS.get(comparison_3d.loc[s, 'Family'], '#333333') for s in top10.index]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(top10)), top10['Sharpe'].values, color=colors, alpha=0.85)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10.index, fontsize=9)
ax.axvline(ML_BAR, color='green', ls='--', lw=1.2, label=f'ML bar ({ML_BAR})')
ax.invert_yaxis()
ax.set_xlabel('OOS Sharpe')
ax.set_title('3d: Top-10 Strategies by Sharpe')
patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()
           if f in comparison_3d['Family'].values]
ax.legend(handles=patches + [plt.Line2D([0], [0], color='green', ls='--', lw=1.2,
          label=f'ML bar ({ML_BAR})')], fontsize=8, loc='lower right')
ax.grid(alpha=0.3)
for i, v in enumerate(top10['Sharpe']):
    ax.text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(fig_dir / 'top10_sharpe_3d.png', dpi=150)
plt.show()

print('Saved 4 figures to', fig_dir)

In [ ]:
print('=' * 60)
print(f'Session 3d Headline  ({RUN_MODE.upper()} mode)')
print('=' * 60)

ML_BAR = 2.579
dw_only   = comparison_3d[comparison_3d['Family'] == 'DL (direct-weight)']
best_dw_name   = dw_only.index[0]
best_dw_sharpe = float(dw_only.iloc[0]['Sharpe'])
diff_dw = best_dw_sharpe - ML_BAR

dl_families = {'DL (single-fit)', 'DL (ensemble)', 'DL (weighted ensemble)', 'DL (direct-weight)'}
dl_all = comparison_3d[comparison_3d['Family'].isin(dl_families)]
best_dl_any_name   = dl_all.index[0]
best_dl_any_sharpe = float(dl_all.iloc[0]['Sharpe'])

best_overall_name   = comparison_3d.index[0]
best_overall_sharpe = float(comparison_3d.iloc[0]['Sharpe'])

print(f'Best DW strategy : {best_dw_name}')
print(f'DW Sharpe        : {best_dw_sharpe:.3f}')
print(f'ML bar           : {ML_BAR:.3f}')
print(f'DW verdict       : {"CLEARED" if diff_dw > 0 else "MISSED"} by {abs(diff_dw):.3f}')
print(f'DW rank          : {comparison_3d.index.get_loc(best_dw_name) + 1} of {len(comparison_3d)}')
print('-----')
print(f'Best DL (any)    : {best_dl_any_name}  ({best_dl_any_sharpe:.3f})')
print(f'Best overall     : {best_overall_name}  ({best_overall_sharpe:.3f})')
print('=' * 60)

print(f'\nStability (OOS Sharpe across {n_seeds} seeds):')
for name, sharpes in seed_sharpes_by_config.items():
    print(f'  {name}: mean {np.mean(sharpes):.3f} ± {np.std(sharpes):.3f}')

print(f'\nComparison rows : {len(comparison_3d)}')
print(f'Stability rows  : {len(stability_table)}')
print(f'Results dir     : {out_dir}')

In [ ]:
if IS_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive('results_cuda_3d', 'zip', str(out_dir))
    print('Triggering download of results_cuda_3d.zip ...')
    files.download('results_cuda_3d.zip')
    print('Done. Extract into ~/Projects/next-gen-aiam/results/cuda/ on your Mac:')
    print('  unzip -o ~/Downloads/results_cuda_3d.zip -d results/cuda/')
else:
    print(f'Local run complete. Results in: {out_dir}')
    import subprocess
    r = subprocess.run(['ls', '-lh', str(out_dir)], capture_output=True, text=True)
    print(r.stdout)